In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [32]:
ls_train = pd.read_csv('train.csv',low_memory=False)
ls_test = pd.read_csv('test.csv')

In [4]:
ls_train.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [5]:
cols_to_reomve = ['ID','Customer_ID','Name','SSN','Month']

In [33]:
ls_train = ls_train.drop(columns=cols_to_reomve)
ls_test = ls_test.drop(columns=cols_to_reomve)

#### Feature Engineer

In [34]:
# Extract years and months
ls_train[['Years', 'Months']] = ls_train['Credit_History_Age'] \
    .str.extract(r'(\d+)\s*Years?\s*and\s*(\d+)\s*Months?') \
    .astype(float)
ls_train['Years'] = pd.to_numeric(ls_train['Years'], errors='coerce')
ls_train['Months'] = pd.to_numeric(ls_train['Months'], errors='coerce')
# Convert to total months
ls_train['Credit_History_Months'] = ls_train['Years'] * 12 +ls_train['Months']

# Drop helper columns
ls_train.drop(columns=['Years', 'Months','Credit_History_Age'], inplace=True)

In [35]:
# Extract years and months
ls_test[['Years', 'Months']] = ls_test['Credit_History_Age'] \
    .str.extract(r'(\d+)\s*Years?\s*and\s*(\d+)\s*Months?') \
    .astype(float)

ls_test['Years'] = pd.to_numeric(ls_test['Years'], errors='coerce')
ls_test['Months'] = pd.to_numeric(ls_test['Months'], errors='coerce')
# Convert to total months
ls_test['Credit_History_Months'] = ls_test['Years'] * 12 +ls_test['Months']

# Drop helper columns
ls_test.drop(columns=['Years', 'Months','Credit_History_Age'], inplace=True)


In [39]:
object_cols = ls_train.select_dtypes(include=['object']).columns

In [43]:
cols_to_numeric = [
    'Age', 'Annual_Income', 'Num_of_Loan',
    'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
    'Outstanding_Debt',
    'Amount_invested_monthly', 'Monthly_Balance'
]

ls_train[cols_to_numeric] = ls_train[cols_to_numeric].apply(lambda x : x.str.replace('_', ''))

ls_test[cols_to_numeric] = ls_test[cols_to_numeric].apply(lambda x : x.str.replace('_', ''))

ls_train[cols_to_numeric] = ls_train[cols_to_numeric].apply(pd.to_numeric, errors='coerce')
ls_test[cols_to_numeric] = ls_test[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

In [44]:
cleaned_ls_train = ls_train
cleaned_ls_test = ls_test

In [45]:
object_cols = ls_train.select_dtypes(include=['object']).columns
cleaned_ls_train[object_cols] = ls_train[object_cols].apply(lambda x: x.str.strip())

In [46]:
object_cols = ls_test.select_dtypes(include=['object']).columns
cleaned_ls_test[object_cols] = ls_test[object_cols].apply(lambda x: x.str.strip())

In [47]:
cleaned_ls_train.isnull().sum()

Age                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary       15002
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment       7002
Changed_Credit_Limit         2091
Num_Credit_Inquiries         1965
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly      4479
Payment_Behaviour               0
Monthly_Balance              1200
Credit_Score                    0
Credit_History_Months        9030
dtype: int64

In [48]:
cleaned_ls_train['Age'] = cleaned_ls_train['Age'].fillna(cleaned_ls_train['Age'].median())

In [49]:
cleaned_ls_test['Age'] = cleaned_ls_test['Age'].fillna(cleaned_ls_test['Age'].median())

In [ ]:
cleaned_ls_train['Annual_Income'] = cleaned_ls_train.groupby('Occupation')['Annual_Income'] \
    .transform(lambda x: x.fillna(x.median()))

In [ ]:
cleaned_ls_test['Annual_Income'] = cleaned_ls_test.groupby('Occupation')['Annual_Income'] \
    .transform(lambda x: x.fillna(x.median()))

In [50]:
cleaned_ls_train['Num_of_Loan'] = cleaned_ls_train['Num_of_Loan'].fillna(0)
cleaned_ls_test['Num_of_Loan'] = cleaned_ls_test['Num_of_Loan'].fillna(0)

In [58]:
cleaned_ls_train['Amount_invested_monthly'] = cleaned_ls_train['Amount_invested_monthly'].fillna(0)
cleaned_ls_test['Amount_invested_monthly'] = cleaned_ls_test['Amount_invested_monthly'].fillna(0)

In [59]:
cleaned_ls_train['Monthly_Balance'] = cleaned_ls_train['Monthly_Balance'].fillna(cleaned_ls_train['Monthly_Balance'].median())
cleaned_ls_test['Monthly_Balance'] = cleaned_ls_test['Monthly_Balance'].fillna(cleaned_ls_test['Monthly_Balance'].median())

In [67]:
cleaned_ls_train['Num_of_Delayed_Payment'] = cleaned_ls_train['Num_of_Delayed_Payment'].fillna(cleaned_ls_train['Num_of_Delayed_Payment'].median())
cleaned_ls_test['Num_of_Delayed_Payment'] = cleaned_ls_test['Num_of_Delayed_Payment'].fillna(cleaned_ls_test['Num_of_Delayed_Payment'].median())

In [68]:
cleaned_ls_train.isnull().sum()

Age                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary           0
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment          0
Changed_Credit_Limit         2091
Num_Credit_Inquiries         1965
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly         0
Payment_Behaviour               0
Monthly_Balance                 0
Credit_Score                    0
Credit_History_Months           0
dtype: int64

In [61]:
cleaned_ls_train['Monthly_Inhand_Salary'] = cleaned_ls_train['Annual_Income'] / 12
cleaned_ls_test['Monthly_Inhand_Salary'] = cleaned_ls_test['Annual_Income'] / 12

In [62]:
cleaned_ls_train['Credit_History_Months'] = cleaned_ls_train['Credit_History_Months'].fillna(0)
cleaned_ls_test['Credit_History_Months'] = cleaned_ls_test['Credit_History_Months'].fillna(0)

In [69]:
cleaned_ls_train.isnull().sum()

Age                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary           0
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment          0
Changed_Credit_Limit         2091
Num_Credit_Inquiries         1965
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly         0
Payment_Behaviour               0
Monthly_Balance                 0
Credit_Score                    0
Credit_History_Months           0
dtype: int64

In [70]:
cleaned_ls_train.dropna(inplace=True)
cleaned_ls_test.dropna(inplace=True)

In [73]:
cleaned_ls_train['Credit_Score_Encoded'] = cleaned_ls_train['Credit_Score'].map({
    'Poor':0,
    'Standard':1,
    'Good':2
})

In [75]:
cleaned_ls_train = cleaned_ls_train.drop(columns=['Credit_Score'])

In [77]:
object_cols = cleaned_ls_train.select_dtypes(include=['object']).columns

In [78]:
cleaned_ls_train[object_cols]

,Occupation,Type_of_Loan,Credit_Mix,Payment_of_Min_Amount,Payment_Behaviour
0,Scientist,"Auto Loan, Credit-Builder Loan, Personal Loan,...",_,No,High_spent_Small_value_payments
1,Scientist,"Auto Loan, Credit-Builder Loan, Personal Loan,...",Good,No,Low_spent_Large_value_payments
3,Scientist,"Auto Loan, Credit-Builder Loan, Personal Loan,...",Good,No,Low_spent_Small_value_payments
4,Scientist,"Auto Loan, Credit-Builder Loan, Personal Loan,...",Good,No,High_spent_Medium_value_payments
5,Scientist,"Auto Loan, Credit-Builder Loan, Personal Loan,...",Good,No,!@9#%8
...,...,...,...,...,...
99995,Mechanic,"Auto Loan, and Student Loan",_,No,High_spent_Large_value_payments
99996,Mechanic,"Auto Loan, and Student Loan",_,No,High_spent_Medium_value_payments
99997,Mechanic,"Auto Loan, and Student Loan",Good,No,High_spent_Large_value_payments
99998,Mechanic,"Auto Loan, and Student Loan",Good,No,Low_spent_Large_value_payments


In [ ]:
df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)
df['Credit_Mix'] = df['Credit_Mix'].fillna('Unknown')

In [79]:
cleaned_ls_train['Payment_Behaviour'].value_counts()

Payment_Behaviour
Low_spent_Small_value_payments      22214
High_spent_Medium_value_payments    14752
Low_spent_Medium_value_payments     11726
High_spent_Large_value_payments     11393
High_spent_Small_value_payments      9639
Low_spent_Large_value_payments       8894
!@9#%8                               6441
Name: count, dtype: int64

In [80]:
valid_payment_patterns = [
    'Low_spent_Small_value_payments',
    'High_spent_Medium_value_payments',
    'Low_spent_Medium_value_payments',
    'High_spent_Large_value_payments',
    'High_spent_Small_value_payments',
    'Low_spent_Large_value_payments'
]

cleaned_ls_train.loc[~cleaned_ls_train['Payment_Behaviour'].isin(valid_payment_patterns),
       'Payment_Behaviour'] = 'Unknown'

cleaned_ls_test.loc[~cleaned_ls_test['Payment_Behaviour'].isin(valid_payment_patterns),
       'Payment_Behaviour'] = 'Unknown'

In [81]:
cleaned_ls_train['Credit_Mix'].value_counts()

Credit_Mix
Standard    31208
Good        18529
Bad         18229
_           17093
Name: count, dtype: int64

In [ ]:
cleaned_ls_train['Credit_Mix'] = cleaned_ls_train['Credit_Mix'].replace('_', np.nan)
cleaned_ls_train['Credit_Mix'] = cleaned_ls_train['Credit_Mix'].fillna('Unknown')


In [82]:
cleaned_ls_test['Credit_Mix'] = cleaned_ls_test['Credit_Mix'].replace('_', np.nan)
cleaned_ls_test['Credit_Mix'] = cleaned_ls_test['Credit_Mix'].fillna('Unknown')


In [84]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

le = LabelEncoder()

for col in cat_cols:
    cleaned_ls_train[col] = le.fit_transform(cleaned_ls_train[col])
    cleaned_ls_test[col] = le.fit_transform(cleaned_ls_test[col])

In [87]:
cleaned_ls_train.drop(columns='Type_of_Loan',inplace=True)
cleaned_ls_test.drop(columns='Type_of_Loan',inplace=True)

In [91]:
corr = cleaned_ls_train.corr()['Credit_Score_Encoded'].sort_values(ascending=False)

In [92]:
corr

Credit_Score_Encoded        1.000000
Credit_History_Months       0.318211
Credit_Mix                  0.114536
Credit_Utilization_Ratio    0.038475
Amount_invested_monthly     0.010260
Monthly_Inhand_Salary       0.007428
Annual_Income               0.007428
Total_EMI_per_month         0.002691
Age                         0.002524
Monthly_Balance            -0.002285
Num_of_Delayed_Payment     -0.003191
Interest_Rate              -0.003981
Num_of_Loan                -0.008754
Num_Credit_Card            -0.009066
Num_Bank_Accounts          -0.010345
Occupation                 -0.011849
Num_Credit_Inquiries       -0.013149
Payment_Behaviour          -0.106953
Changed_Credit_Limit       -0.155021
Payment_of_Min_Amount      -0.276325
Outstanding_Debt           -0.387593
Delay_from_due_date        -0.429586
Name: Credit_Score_Encoded, dtype: float64

In [93]:
from sklearn.model_selection import train_test_split
X = cleaned_ls_train.drop(columns=['Credit_Score_Encoded'])
y= cleaned_ls_train['Credit_Score_Encoded']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

- Which model works better here: **Logistic Regression, Random Forest, XGBoost, or LightGBM**?  

In [94]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

log_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Logistic Regression Accuracy: 0.5764166470726546
              precision    recall  f1-score   support

           0       0.60      0.64      0.62      5306
           1       0.72      0.47      0.57      8941
           2       0.40      0.79      0.53      2765

    accuracy                           0.58     17012
   macro avg       0.57      0.64      0.57     17012
weighted avg       0.63      0.58      0.58     17012



In [95]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.7847989654361627
              precision    recall  f1-score   support

           0       0.78      0.81      0.80      5306
           1       0.80      0.81      0.80      8941
           2       0.74      0.67      0.70      2765

    accuracy                           0.78     17012
   macro avg       0.77      0.76      0.77     17012
weighted avg       0.78      0.78      0.78     17012



In [96]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

XGBoost Accuracy: 0.7359510933458735
              precision    recall  f1-score   support

           0       0.76      0.71      0.74      5306
           1       0.76      0.77      0.77      8941
           2       0.62      0.66      0.64      2765

    accuracy                           0.74     17012
   macro avg       0.71      0.72      0.71     17012
weighted avg       0.74      0.74      0.74     17012




What we observed:

Logistic Regression → lower

Random Forest → strong

XGBoost → 72.7%

Random Forest is slightly outperforming in our case.

This is common when:

Dataset is tabular

Moderate feature complexity

Limited feature engineering
We benchmarked Logistic Regression, Random Forest, and XGBoost. Random Forest provided the best balance between precision and recall across all three credit classes, achieving approximately 73% accuracy. Further improvements can be achieved through advanced feature engineering and hyperparameter tuning